In [ ]:
############################################################
# 0. Imports and setup
############################################################
# In Kaggle, you may need:
!pip install -q scikit-image pydensecrf2

import os
from glob import glob
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
import pydensecrf.densecrf as dcrf
from pydensecrf.utils import unary_from_softmax
from skimage import measure

# ==========================================================
# RUN CONFIGURATION
# Set to "both", "lim_cd", or "levir_cdpp" to optimize training time
TARGET_DATASET = "levir_cdpp"  
# ==========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

############################################################
# 1. Dataset configuration for both LIM-CD and LEVIR-CD++
############################################################

DATASETS = [
    {
        "name": "lim_cd",
        "root": "/kaggle/input/datasets/ravat1920/lim-cd/lim-cd",  # adjust if needed
        "t1_subdir": "t1",
        "t2_subdir": "t2",
        "mask_subdir": "label",    # change to "mask" if your folder is named that
        "use_patching": False,     # <-- LIM-CD uses standard resizing
        "img_size": 512,
        "batch_size": 4,
        "num_epochs": 80,
        "lr": 1e-4,
        "num_workers": 2,
        "focal_gamma": 2.0,
        "focal_alpha": 0.75,
        "focal_weight": 0.7,
        "deep_sup_weights": [1.0, 0.4, 0.2],
        "val_split": "val",
    },
    {
        "name": "levir_cdpp",
        "root": "/kaggle/input/datasets/mdrifaturrahman33/levir-cd/LEVIR CD",  # adjust to your LEVIR-CD++ root
        "t1_subdir": "A",
        "t2_subdir": "B",
        "mask_subdir": "label",
        "use_patching": True,      # <-- LEVIR-CD uses the new dynamic patching
        "tile_size": 256,
        "stride": 256,
        "min_change_pixels": 10,
        "neg_ratio": 1.5,
        "batch_size": 32,
        "num_epochs": 50,
        "lr": 1e-4,
        "num_workers": 2,
        "focal_gamma": 1.5,
        "focal_alpha": 0.7,
        "focal_weight": 0.5,
        "deep_sup_weights": [1.0, 0.3, 0.1],
        "val_split": "val",
    },
]

PRINT_INTERVAL = 50

############################################################
# 2. Dataset Classes (Standard & Patched)
############################################################

class BiTemporalDataset(Dataset):
    """
    Generic T1/T2/mask dataset for standard whole-image resizing.
    """
    def __init__(self,
                 root_dir,
                 split="train",
                 img_size=256,
                 use_imagenet_norm=True,
                 augment=True,
                 t1_subdir="t1",
                 t2_subdir="t2",
                 mask_subdir="label"):
        self.root_dir = Path(root_dir)
        self.split = split
        self.img_size = img_size
        self.augment = augment and (split == "train")

        split_dir = self.root_dir / split
        self.t1_paths = sorted(glob(str(split_dir / t1_subdir / "*")))
        self.t2_paths = sorted(glob(str(split_dir / t2_subdir / "*")))
        self.mask_paths = sorted(glob(str(split_dir / mask_subdir / "*")))

        assert len(self.t1_paths) == len(self.t2_paths) == len(self.mask_paths), \
            f"[{root_dir}/{split}] Mismatch: t1={len(self.t1_paths)}, t2={len(self.t2_paths)}, mask={len(self.mask_paths)}"

        base_tfms = [
            transforms.Resize((img_size, img_size)),
        ]

        if self.augment:
            base_tfms += [
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(degrees=10),
                transforms.ColorJitter(
                    brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1
                ),
            ]

        base_tfms += [transforms.ToTensor()]
        self.base_transform = transforms.Compose(base_tfms)

        if use_imagenet_norm:
            self.norm = transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            )
        else:
            self.norm = None

    def __len__(self):
        return len(self.t1_paths)

    def _load_image(self, path, mode="RGB"):
        return Image.open(path).convert(mode)

    def __getitem__(self, idx):
        t1_path = self.t1_paths[idx]
        t2_path = self.t2_paths[idx]
        mask_path = self.mask_paths[idx]

        t1 = self._load_image(t1_path, "RGB")
        t2 = self._load_image(t2_path, "RGB")
        mask = self._load_image(mask_path, "L")

        seed = np.random.randint(0, 10_000)
        torch.manual_seed(seed)
        t1 = self.base_transform(t1)
        torch.manual_seed(seed)
        t2 = self.base_transform(t2)
        torch.manual_seed(seed)
        mask = self.base_transform(mask)

        if self.norm is not None:
            t1 = self.norm(t1)
            t2 = self.norm(t2)

        mask = (mask > 0.5).float()
        return t1, t2, mask


class PatchedChangeDataset(Dataset):
    """
    Dynamic patching dataset for large-scale images (e.g., LEVIR-CD).
    """
    def __init__(self, root_dir, augment=False,
                 tile_size=256, stride=256,
                 min_change_pixels=10,
                 neg_ratio=1.5,
                 t1_subdir="A", t2_subdir="B", mask_subdir="label"):
        self.root_dir  = root_dir
        self.augment   = augment
        self.tile_size = tile_size
        self.stride    = stride

        self.t1_subdir = t1_subdir
        self.t2_subdir = t2_subdir
        self.mask_subdir = mask_subdir

        self.A = sorted(os.listdir(os.path.join(root_dir, self.t1_subdir)))
        self.B = sorted(os.listdir(os.path.join(root_dir, self.t2_subdir)))
        self.Y = sorted(os.listdir(os.path.join(root_dir, self.mask_subdir)))

        if len(self.A) == 0:
            raise RuntimeError(f"Found 0 images in {os.path.join(root_dir, self.t1_subdir)}")

        _probe = Image.open(os.path.join(root_dir, self.t1_subdir, self.A[0]))
        full_W, full_H = _probe.size
        _probe.close()

        raw_patches = []
        if full_H == tile_size and full_W == tile_size:
            for i in range(len(self.A)):
                raw_patches.append((i, 0, 0))
        else:
            for i in range(len(self.A)):
                for r in range(0, full_H - tile_size + 1, stride):
                    for c in range(0, full_W - tile_size + 1, stride):
                        raw_patches.append((i, r, c))

        split_name = os.path.basename(root_dir)

        # ── neg_ratio=None → val/test: keep ALL patches, no filtering ──
        if neg_ratio is None:
            self.patches = raw_patches
            print(f"  [{split_name}] {len(self.A)} images → "
                  f"{len(self.patches)} patches (NO filtering — full distribution)")
        else:
            # ── Train: controlled negative sampling ──
            print(f"  [{split_name}] Scanning {len(raw_patches)} patches...", end=" ")

            change_patches    = []
            no_change_patches = []

            _mask_cache_idx = -1
            _mask_cache_arr = None

            for (img_idx, r, c) in raw_patches:
                if img_idx != _mask_cache_idx:
                    mask_path = os.path.join(root_dir, self.mask_subdir, self.Y[img_idx])
                    _mask_cache_arr = np.array(Image.open(mask_path).convert("L"))
                    _mask_cache_idx = img_idx

                tile_mask = _mask_cache_arr[r:r + tile_size, c:c + tile_size]
                n_change  = (tile_mask > 127).sum()

                if n_change >= min_change_pixels:
                    change_patches.append((img_idx, r, c))
                else:
                    no_change_patches.append((img_idx, r, c))

            # REFINEMENT 3: seed before shuffle for reproducibility
            np.random.seed(42)
            np.random.shuffle(no_change_patches)

            # REFINEMENT 1: neg_ratio=1.5 (not 1.0) for realistic distribution
            max_neg  = int(len(change_patches) * neg_ratio)
            kept_neg = no_change_patches[:max_neg]

            self.patches = change_patches + kept_neg
            np.random.seed(42)
            np.random.shuffle(self.patches)

            print(f"done")
            print(f"  [{split_name}] Change patches : {len(change_patches)}")
            print(f"  [{split_name}] No-change kept : {len(kept_neg)} "
                  f"(ratio={neg_ratio}× → max {max_neg})")
            print(f"  [{split_name}] Total used     : {len(self.patches)} "
                  f"/ {len(raw_patches)} raw\n")

        self.to_tensor_img  = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225])
        ])
        self.to_tensor_mask = transforms.ToTensor()

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        img_idx, r, c = self.patches[idx]
        box = (c, r, c + self.tile_size, r + self.tile_size)

        img1 = Image.open(os.path.join(self.root_dir, self.t1_subdir, self.A[img_idx])).convert("RGB").crop(box)
        img2 = Image.open(os.path.join(self.root_dir, self.t2_subdir, self.B[img_idx])).convert("RGB").crop(box)
        mask = Image.open(os.path.join(self.root_dir, self.mask_subdir, self.Y[img_idx])).convert("L").crop(box)

        if self.augment:
            if torch.rand(1) < 0.5:
                img1 = TF.hflip(img1); img2 = TF.hflip(img2); mask = TF.hflip(mask)
            if torch.rand(1) < 0.5:
                img1 = TF.vflip(img1); img2 = TF.vflip(img2); mask = TF.vflip(mask)
            if torch.rand(1) < 0.3:
                angle = transforms.RandomRotation.get_params([-30, 30])
                img1 = TF.rotate(img1, angle, interpolation=InterpolationMode.BILINEAR)
                img2 = TF.rotate(img2, angle, interpolation=InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
            if torch.rand(1) < 0.4:
                img1 = TF.adjust_brightness(img1, torch.empty(1).uniform_(0.75, 1.25).item())
                img1 = TF.adjust_contrast(img1,   torch.empty(1).uniform_(0.75, 1.25).item())
                img2 = TF.adjust_brightness(img2, torch.empty(1).uniform_(0.75, 1.25).item())
                img2 = TF.adjust_contrast(img2,   torch.empty(1).uniform_(0.75, 1.25).item())

        img1 = self.to_tensor_img(img1)
        img2 = self.to_tensor_img(img2)
        mask = self.to_tensor_mask(mask)
        mask = (mask > 0.5).float()

        return img1, img2, mask


def get_dataloaders(cfg):
    if cfg.get("use_patching", False):
        # Use dynamic patching logic
        train_ds = PatchedChangeDataset(
            root_dir=os.path.join(cfg["root"], "train"),
            augment=True,
            tile_size=cfg.get("tile_size", 256),
            stride=cfg.get("stride", 256),
            min_change_pixels=cfg.get("min_change_pixels", 10),
            neg_ratio=cfg.get("neg_ratio", 1.5),
            t1_subdir=cfg["t1_subdir"],
            t2_subdir=cfg["t2_subdir"],
            mask_subdir=cfg["mask_subdir"]
        )
        val_ds = PatchedChangeDataset(
            root_dir=os.path.join(cfg["root"], cfg["val_split"]),
            augment=False,
            tile_size=cfg.get("tile_size", 256),
            stride=cfg.get("stride", 256),
            neg_ratio=None, # full distribution for validation
            t1_subdir=cfg["t1_subdir"],
            t2_subdir=cfg["t2_subdir"],
            mask_subdir=cfg["mask_subdir"]
        )
    else:
        # Use standard resizing logic
        train_ds = BiTemporalDataset(
            cfg["root"], split="train",
            img_size=cfg["img_size"],
            augment=True,
            t1_subdir=cfg["t1_subdir"],
            t2_subdir=cfg["t2_subdir"],
            mask_subdir=cfg["mask_subdir"],
        )
        val_ds   = BiTemporalDataset(
            cfg["root"], split=cfg["val_split"],
            img_size=cfg["img_size"],
            augment=False,
            t1_subdir=cfg["t1_subdir"],
            t2_subdir=cfg["t2_subdir"],
            mask_subdir=cfg["mask_subdir"],
        )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=cfg["num_workers"],
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=cfg["num_workers"],
        pin_memory=True,
    )

    print(f"[{cfg['name']}] Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    return train_loader, val_loader

############################################################
# 3. Model: Siamese ResNet50 + fusion + UNet decoder
############################################################

# class ResNet50Encoder(nn.Module):
#     def __init__(self, pretrained=True):
#         super().__init__()
#         if pretrained:
#             backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
#         else:
#             backbone = models.resnet50(weights=None)

#         self.conv1 = backbone.conv1
#         self.bn1 = backbone.bn1
#         self.relu = backbone.relu
#         self.maxpool = backbone.maxpool

#         self.layer1 = backbone.layer1  # 1/4,  256
#         self.layer2 = backbone.layer2  # 1/8,  512
#         self.layer3 = backbone.layer3  # 1/16, 1024
#         self.layer4 = backbone.layer4  # 1/32, 2048

#         self.proj1 = nn.Conv2d(256,  64,  kernel_size=1)
#         self.proj2 = nn.Conv2d(512,  128, kernel_size=1)
#         self.proj3 = nn.Conv2d(1024, 256, kernel_size=1)
#         self.proj4 = nn.Conv2d(2048, 512, kernel_size=1)

#     def forward(self, x):
#         x = self.conv1(x)
#         x = self.bn1(x)
#         x = self.relu(x)
#         x = self.maxpool(x)

#         x1 = self.layer1(x)
#         x2 = self.layer2(x1)
#         x3 = self.layer3(x2)
#         x4 = self.layer4(x3)

#         f1 = self.proj1(x1)
#         f2 = self.proj2(x2)
#         f3 = self.proj3(x3)
#         f4 = self.proj4(x4)

#         return [f1, f2, f3, f4]

class ResNet18Encoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        if pretrained:
            backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        else:
            backbone = models.resnet18(weights=None)

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        # ResNet18 naturally outputs the exact channel depths we need!
        self.layer1 = backbone.layer1  # 1/4,  64
        self.layer2 = backbone.layer2  # 1/8,  128
        self.layer3 = backbone.layer3  # 1/16, 256
        self.layer4 = backbone.layer4  # 1/32, 512

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)

        return [f1, f2, f3, f4]


class CrossAttention2D(nn.Module):
    def __init__(self, channels=512, num_heads=8):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=channels,
                                         num_heads=num_heads,
                                         batch_first=False)

    def forward(self, fa, fb):
        B, C, H, W = fa.shape
        fa_seq = fa.view(B, C, H * W).permute(0, 2, 1)
        fb_seq = fb.view(B, C, H * W).permute(0, 2, 1)

        q = fa_seq.permute(1, 0, 2)
        k = fb_seq.permute(1, 0, 2)
        v = fb_seq.permute(1, 0, 2)

        attn_out, _ = self.mha(q, k, v)
        attn_out = attn_out.permute(1, 2, 0).view(B, C, H, W)
        return attn_out


def fusion_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.size(-1) != skip.size(-1) or x.size(-2) != skip.size(-2):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x


class SiameseUNetChangeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        # self.encoder = ResNet50Encoder(pretrained=True)
        self.encoder = ResNet18Encoder(pretrained=True)

        self.fuse1 = fusion_block(in_ch=3 * 64,  out_ch=64)
        self.fuse2 = fusion_block(in_ch=3 * 128, out_ch=128)
        self.fuse3 = fusion_block(in_ch=3 * 256, out_ch=256)
        self.fuse4 = fusion_block(in_ch=3 * 512, out_ch=512)

        self.cross_attn = CrossAttention2D(channels=512, num_heads=8)

        self.up3 = UpBlock(in_ch=512, skip_ch=256, out_ch=256)
        self.up2 = UpBlock(in_ch=256, skip_ch=128, out_ch=128)
        self.up1 = UpBlock(in_ch=128, skip_ch=64,  out_ch=64)

        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )

        # main + two aux heads, all from 64-channel branch (d1 / upsampled d1)
        self.head_main = nn.Conv2d(16, 1, kernel_size=1)
        self.head_aux1 = nn.Conv2d(64, 1, kernel_size=1)   # 1/4 scale
        self.head_aux2 = nn.Conv2d(64, 1, kernel_size=1)   # 1/2 scale (from upsampled d1)

    def forward(self, x1, x2):
        a1, a2, a3, a4 = self.encoder(x1)
        b1, b2, b3, b4 = self.encoder(x2)

        def fuse(a, b):
            diff = torch.abs(a - b)
            return torch.cat([a, b, diff], dim=1)

        f1 = self.fuse1(fuse(a1, b1))   # (B,64, H/4,  W/4)
        f2 = self.fuse2(fuse(a2, b2))   # (B,128,H/8,  W/8)
        f3 = self.fuse3(fuse(a3, b3))   # (B,256,H/16, W/16)
        f4 = self.fuse4(fuse(a4, b4))   # (B,512,H/32, W/32)

        f4_attn = self.cross_attn(f4, f4)

        d3 = self.up3(f4_attn, f3)      # 1/16
        d2 = self.up2(d3, f2)           # 1/8
        d1 = self.up1(d2, f1)           # 1/4

        # aux1 at 1/4
        aux1_logits = self.head_aux1(d1)

        # aux2 at 1/2 (upsampled d1, still 64 channels)
        up_half = F.interpolate(d1, scale_factor=2, mode="bilinear", align_corners=False)
        aux2_logits = self.head_aux2(up_half)

        # main at full
        x = self.up0(d1)
        main_logits = self.head_main(x)

        p_main = torch.sigmoid(main_logits)
        p_aux1 = torch.sigmoid(aux1_logits)
        p_aux2 = torch.sigmoid(aux2_logits)

        return p_main, p_aux1, p_aux2

############################################################
# 4. Loss & metrics
############################################################

class BCEDiceFocalLoss(nn.Module):
    def __init__(self,
                 focal_gamma=2.0,
                 focal_alpha=0.75,
                 focal_weight=0.5,
                 eps=1e-6):
        super().__init__()
        self.bce = nn.BCELoss()
        self.focal_gamma = focal_gamma
        self.focal_alpha = focal_alpha
        self.focal_weight = focal_weight
        self.eps = eps

    def dice_loss(self, pred, target):
        pred_flat = pred.view(pred.size(0), -1)
        target_flat = target.view(target.size(0), -1)

        intersection = (pred_flat * target_flat).sum(1)
        union = pred_flat.sum(1) + target_flat.sum(1)

        dice = 1 - (2 * intersection + self.eps) / (union + self.eps)
        return dice.mean()

    def focal_loss(self, pred, target):
        p = pred.clamp(self.eps, 1.0 - self.eps)
        ce = -(target * torch.log(p) + (1 - target) * torch.log(1 - p))
        pt = torch.where(target == 1, p, 1 - p)
        w = self.focal_alpha * (1 - pt) ** self.focal_gamma
        fl = (w * ce).mean()
        return fl

    def forward(self, pred, target):
        bce = self.bce(pred, target)
        dice = self.dice_loss(pred, target)
        focal = self.focal_loss(pred, target)
        return bce + dice + self.focal_weight * focal


def multiscale_loss(loss_fn, preds, target, weights):
    p_main, p_aux1, p_aux2 = preds
    w_main, w_aux1, w_aux2 = weights

    L = 0.0
    L += w_main * loss_fn(p_main, target)

    if w_aux1 > 0:
        t1 = F.interpolate(target, size=p_aux1.shape[-2:], mode="nearest")
        L += w_aux1 * loss_fn(p_aux1, t1)

    if w_aux2 > 0:
        t2 = F.interpolate(target, size=p_aux2.shape[-2:], mode="nearest")
        L += w_aux2 * loss_fn(p_aux2, t2)

    return L


def compute_segmentation_metrics(pred, target, eps=1e-6):
    pred = pred.view(-1)
    target = target.view(-1)

    tp = (pred * target).sum().item()
    fp = (pred * (1 - target)).sum().item()
    fn = ((1 - pred) * target).sum().item()

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)

    intersection = tp
    union = tp + fp + fn
    iou = intersection / (union + eps)

    return precision, recall, f1, iou

############################################################
# 5. Train / eval loops
############################################################

def train_one_epoch(model, loader, optimizer, loss_fn, device, epoch, deep_sup_weights, dataset_name):
    model.train()
    running_loss = 0.0

    for step, (x1, x2, y) in enumerate(loader, start=1):
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(x1, x2)
        loss = multiscale_loss(loss_fn, preds, y, deep_sup_weights)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x1.size(0)

        if step % PRINT_INTERVAL == 0:
            print(f"[{dataset_name}] Epoch {epoch} | Step {step}/{len(loader)} | "
                  f"Batch Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss

############################################################
# Post-Processing Pipeline (CRF + Object Aggregation)
############################################################

def post_process_mask(t2_img_tensor, prob_map_tensor, min_object_area=64):
    """
    Applies Dense CRF for boundary refinement and Connected Components for object aggregation.
    """
    # 1. Un-normalize the T2 image back to 0-255 uint8 format for the CRF bilateral filter
    mean = torch.tensor([0.485, 0.456, 0.406], device=t2_img_tensor.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=t2_img_tensor.device).view(3, 1, 1)
    
    t2_img_np = (t2_img_tensor[0] * std + mean).clamp(0, 1).cpu().numpy()
    t2_img_np = (t2_img_np.transpose(1, 2, 0) * 255.0).astype(np.uint8)
    t2_img_np = np.ascontiguousarray(t2_img_np) 
    
    # 2. Prepare Probabilities for CRF
    p_change = prob_map_tensor[0, 0].cpu().numpy()
    p_no_change = 1.0 - p_change
    probs = np.stack([p_no_change, p_change], axis=0) # Shape: (2, H, W)
    
    # 3. Setup Dense CRF
    H, W = p_change.shape
    d = dcrf.DenseCRF2D(W, H, 2)
    
    U = unary_from_softmax(probs)
    d.setUnaryEnergy(U)
    
    # Smoothness (removes isolated specs)
    d.addPairwiseGaussian(sxy=(3, 3), compat=3, kernel=dcrf.DIAG_KERNEL, normalization=dcrf.NORMALIZE_SYMMETRIC)
    # Appearance (snaps to visual edges in the T2 image)
    d.addPairwiseBilateral(sxy=(50, 50), srgb=(13, 13, 13), rgbim=t2_img_np, compat=10, 
                           kernel=dcrf.DIAG_KERNEL, normalization=dcrf.NORMALIZE_SYMMETRIC)
    
    # 4. Run Inference (5 iterations)
    Q = d.inference(5)
    crf_map = np.argmax(Q, axis=0).reshape((H, W)).astype(np.uint8)
    
    # 5. Object Aggregation (Connected Components)
    labeled_mask = measure.label(crf_map, connectivity=2)
    final_mask = np.zeros_like(crf_map)
    
    for region in measure.regionprops(labeled_mask):
        if region.area >= min_object_area:
            final_mask[labeled_mask == region.label] = 1
            
    return final_mask
    
def evaluate(model, loader, loss_fn, device, deep_sup_weights, thr=0.5, use_post_processing=False):
    model.eval()
    running_loss = 0.0
    all_prec, all_rec, all_f1, all_iou = [], [], [], []

    with torch.no_grad():
        for x1, x2, y in loader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)

            preds = model(x1, x2)
            loss = multiscale_loss(loss_fn, preds, y, deep_sup_weights)
            running_loss += loss.item() * x1.size(0)

            p_main = preds[0]
            
            if use_post_processing:
                # Run the heavy CPU pipeline on each image in the batch
                batch_preds = []
                for i in range(x2.size(0)):
                    clean_mask = post_process_mask(x2[i:i+1], p_main[i:i+1], min_object_area=64)
                    batch_preds.append(torch.from_numpy(clean_mask).to(device))
                pred_bin = torch.stack(batch_preds).unsqueeze(1).float()
            else:
                # Fast GPU thresholding for normal epochs
                pred_bin = (p_main > thr).float()

            precision, recall, f1, iou = compute_segmentation_metrics(pred_bin, y)
            all_prec.append(precision)
            all_rec.append(recall)
            all_f1.append(f1)
            all_iou.append(iou)

    epoch_loss = running_loss / len(loader.dataset)
    metrics = {
        "precision": float(np.mean(all_prec)),
        "recall": float(np.mean(all_rec)),
        "f1": float(np.mean(all_f1)),
        "iou": float(np.mean(all_iou)),
    }
    return epoch_loss, metrics

############################################################
# 6. Train both datasets and compare
############################################################

def train_on_dataset(cfg):
    print(f"\n===== Training on {cfg['name']} =====")
    train_loader, val_loader = get_dataloaders(cfg)

    model = SiameseUNetChangeDetector().to(DEVICE)
    loss_fn = BCEDiceFocalLoss(
        focal_gamma=cfg["focal_gamma"],
        focal_alpha=cfg["focal_alpha"],
        focal_weight=cfg["focal_weight"],
    )
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg["lr"],
                                  weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["num_epochs"], eta_min=1e-6
    )

    best_val_f1 = 0.0
    best_metrics = None
    model_save_path = f"best_model_{cfg['name']}.pth"

    # --- MAIN TRAINING LOOP ---
    for epoch in range(1, cfg["num_epochs"] + 1):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, loss_fn, DEVICE,
            epoch, cfg["deep_sup_weights"], cfg["name"]
        )
        # Fast evaluation without post-processing
        val_loss, val_metrics = evaluate(
            model, val_loader, loss_fn, DEVICE,
            cfg["deep_sup_weights"], thr=0.5, use_post_processing=False
        )

        scheduler.step()

        print(f"[{cfg['name']}] Epoch {epoch}/{cfg['num_epochs']} "
              f"| Train Loss: {train_loss:.4f} "
              f"| Val Loss: {val_loss:.4f} "
              f"| Val F1: {val_metrics['f1']:.4f} "
              f"| Val IoU: {val_metrics['iou']:.4f}")

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_metrics = val_metrics
            torch.save(model.state_dict(), model_save_path)
            print(f"  -> [{cfg['name']}] Saved new best model")

    # --- FINAL POST-PROCESSING EVALUATION ---
    print(f"\n[{cfg['name']}] Training complete. Running final Object-Level Post-Processing...")
    model.load_state_dict(torch.load(model_save_path))
    
    # Note: This will take a few minutes as the CRF runs on the CPU
    _, final_clean_metrics = evaluate(
        model, val_loader, loss_fn, DEVICE,
        cfg["deep_sup_weights"], thr=0.5, use_post_processing=True
    )
    
    print(f"[{cfg['name']}] Final Refined F1: {final_clean_metrics['f1']:.4f} | IoU: {final_clean_metrics['iou']:.4f}")

    # Return the clean metrics for the final printout table
    return final_clean_metrics


if __name__ == "__main__":
    results = {}

    # Filter which datasets to run based on the TARGET_DATASET flag
    datasets_to_run = DATASETS
    if TARGET_DATASET != "both":
        datasets_to_run = [cfg for cfg in DATASETS if cfg["name"] == TARGET_DATASET]

    if not datasets_to_run:
        print(f"Error: No datasets found matching TARGET_DATASET='{TARGET_DATASET}'. Please check your spelling.")
    else:
        for cfg in datasets_to_run:
            metrics = train_on_dataset(cfg)
            results[cfg["name"]] = metrics

        # Print comparison table
        print("\n===== Final Comparison (Validation) =====")
        header = f"{'Dataset':<12} | {'Precision':>9} | {'Recall':>7} | {'F1':>7} | {'IoU':>7}"
        print(header)
        print("-" * len(header))
        for name, m in results.items():
            if m is None:
                continue
            print(f"{name:<12} | {m['precision']:9.4f} | {m['recall']:7.4f} | "
                  f"{m['f1']:7.4f} | {m['iou']:7.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.5 MB/s eta 0:00:0000:01:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

===== Training on levir_cdpp =====
  [train] Scanning 7120 patches... done
  [train] Change patches : 3153
  [train] No-change kept : 3967 (ratio=1.5× → max 4729)
  [train] Total used     : 7120 / 7120 raw

  [val] 64 images → 1024 patches (NO filtering — full distribution)
[levir_cdpp] Train batches: 223, Val batches: 32
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 215MB/s]


[levir_cdpp] Epoch 1 | Step 50/223 | Batch Loss: 2.0383
[levir_cdpp] Epoch 1 | Step 100/223 | Batch Loss: 1.9167
[levir_cdpp] Epoch 1 | Step 150/223 | Batch Loss: 1.7569
[levir_cdpp] Epoch 1 | Step 200/223 | Batch Loss: 1.7173
[levir_cdpp] Epoch 1/50 | Train Loss: 1.8884 | Val Loss: 1.7023 | Val F1: 0.7210 | Val IoU: 0.5958
  -> [levir_cdpp] Saved new best model
[levir_cdpp] Epoch 2 | Step 50/223 | Batch Loss: 1.7075
[levir_cdpp] Epoch 2 | Step 100/223 | Batch Loss: 1.7063
[levir_cdpp] Epoch 2 | Step 150/223 | Batch Loss: 1.5472
[levir_cdpp] Epoch 2 | Step 200/223 | Batch Loss: 1.5942
[levir_cdpp] Epoch 2/50 | Train Loss: 1.6362 | Val Loss: 1.5563 | Val F1: 0.7762 | Val IoU: 0.6691
  -> [levir_cdpp] Saved new best model
[levir_cdpp] Epoch 3 | Step 50/223 | Batch Loss: 1.4427
[levir_cdpp] Epoch 3 | Step 100/223 | Batch Loss: 1.5440
[levir_cdpp] Epoch 3 | Step 150/223 | Batch Loss: 1.4486
[levir_cdpp] Epoch 3 | Step 200/223 | Batch Loss: 1.5070
[levir_cdpp] Epoch 3/50 | Train Loss: 1.516